#Importing the libraries

In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
import joblib

#Load the dataset

In [28]:
df=pd.read_csv("/content/Historical Product Demand.csv")
df=df[['Product_Code','Warehouse','Product_Category','Date','Order_Demand']]
df.head()

,Product_Code,Warehouse,Product_Category,Date,Order_Demand
0,Product_0993,Whse_J,Category_028,2012/7/27,100
1,Product_0979,Whse_J,Category_028,2012/1/19,500
2,Product_0979,Whse_J,Category_028,2012/2/3,500
3,Product_0979,Whse_J,Category_028,2012/2/9,500
4,Product_0979,Whse_J,Category_028,2012/3/2,500


In [29]:
df['Order_Demand']=df['Order_Demand'].astype(str).str.replace('(', '-')
df['Order_Demand']=df['Order_Demand'].str.replace(')', '')
df['Order_Demand']=df['Order_Demand'].astype(float)
df['Date']=pd.to_datetime(df['Date'])
df=df.dropna()
df['Order_Demand']=df['Order_Demand'].apply(lambda x: max(0, x))

In [30]:
df=df.groupby(['Product_Code','Date']).agg({
    'Order_Demand':'sum',
    'Warehouse':'first',
    'Product_Category':'first'
}).reset_index()

In [31]:
df['Order_Demand']=np.log1p(df['Order_Demand'])

In [32]:
df=df.sort_values(by=['Product_Code','Date'])
df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month
df['Day']=df['Date'].dt.day
df['Weekday'] = df['Date'].dt.weekday
df['Lag_1']=df.groupby('Product_Code')['Order_Demand'].shift(1)
df['Lag_2']=df.groupby('Product_Code')['Order_Demand'].shift(2)
df['Lag_3']=df.groupby('Product_Code')['Order_Demand'].shift(3)
df['Rolling_Mean'] = df.groupby('Product_Code')['Order_Demand']\
                      .transform(lambda x: x.shift(1).rolling(3).mean())

df = df.dropna()

#LabelEncoding

In [33]:
le_product=LabelEncoder()
le_warehouse=LabelEncoder()
le_category=LabelEncoder()
df['Product_Code']=le_product.fit_transform(df['Product_Code'])
df['Warehouse']=le_warehouse.fit_transform(df['Warehouse'])
df['Product_Category']=le_category.fit_transform(df['Product_Category'])

In [34]:
X=df[['Product_Code', 'Warehouse', 'Product_Category',
        'Year', 'Month', 'Day', 'Weekday',
        'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean']]
y=df['Order_Demand']
df=df.sort_values('Date')
split_index=int(len(df)*0.8)
X_train=X[:split_index]
X_test=X[split_index:]
y_train=y[:split_index]
y_test=y[split_index:]

#LinearRegression

In [35]:
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)
print("Linear Regression")
print("RMSE:",np.sqrt(mean_squared_error(y_test,y_pred_lr)))
print("R2:",r2_score(y_test,y_pred_lr))

Linear Regression
RMSE: 1.3386054643188547
R2: 0.7741269693054789


#ExtraTreesRegressor

In [36]:
et=ExtraTreesRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
et.fit(X_train, y_train)
y_pred_et = et.predict(X_test)
print("\nExtra Trees")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_et)))
print("R2 Score:", r2_score(y_test, y_pred_et))


Extra Trees
RMSE: 1.2948909380040319
R2 Score: 0.7886386494791888


In [37]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nOptimized Random Forest")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2 Score:", r2_score(y_test, y_pred_rf))


Optimized Random Forest
RMSE: 1.2625249974152828
R2 Score: 0.7990726004510897


#Which Model is best

In [43]:
scores = {
    "Linear Regression": r2_score(y_test, y_pred_lr),
    "Extra Trees": r2_score(y_test, y_pred_et),
    "Random Forest": r2_score(y_test, y_pred_rf)
}
best_model_name=max(scores, key=scores.get)
if best_model_name=="Linear Regression":
    best_model = lr
elif best_model_name == "Extra Trees":
    best_model = et
else:
    best_model = rf
print("\nBest Model:", best_model_name)


Best Model: Random Forest


#Model save

In [39]:
joblib.dump(best_model, "demand_forecast_model.pkl")

['demand_forecast_model.pkl']

In [40]:
model=joblib.load("demand_forecast_model.pkl")
sample=pd.DataFrame([[100,1,5,2015,6,15,2,5.0,5.2,5.1,5.1]],columns=X.columns)
prediction=model.predict(sample)
final_prediction=np.expm1(prediction[0])
print("Predicted Demand:",final_prediction)

Predicted Demand: 252.39537302276582
